In [86]:
# Basic Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns
# Modelling
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression, Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
import warnings

In [87]:
df = pd.read_csv('data/European_Bank.csv')

In [88]:
df.head(5)

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [89]:
categorical_cols = df.select_dtypes(include='object').columns
categorical_cols

Index(['Surname', 'Geography', 'Gender'], dtype='object')

In [90]:
conditions = [
        (df["IsActiveMember"] == 1) & (df["NumOfProducts"] >= 2),   # Active + multi-product
        (df["IsActiveMember"] == 0) & (df["NumOfProducts"] <= 1),   # Inactive + low-product
        (df["IsActiveMember"] == 1) & (df["NumOfProducts"] == 1),   # Active but low-product
        (df["IsActiveMember"] == 0) & (df["Balance"] > df["Balance"].median()),  # Inactive high-balance
    ]

labels = [
        "Active_Engaged",
        "Inactive_Disengaged",
        "Active_LowProduct",
        "Inactive_HighBalance",
    ]

df["EngagementSegment"] = np.select(conditions, labels, default="Other")
 
    # --- 1.2 Activity Score (weighted composite) ---
df["ActivityScore"] = (
        df["IsActiveMember"] * 3 +
        df["HasCrCard"] * 1 +
        (df["NumOfProducts"] >= 2).astype(int) * 2
    )
 
    # --- 1.3 Tenure Bands ---
df["TenureBand"] = pd.cut(
        df["Tenure"],
        bins=[-1, 1, 3, 6, 10],
        labels=["New (0-1y)", "Early (2-3y)", "Mid (4-6y)", "Loyal (7y+)"]
    )
 
    # --- 1.4 Is Long-Tenure Active ---
df["IsLongTenureActive"] = (
        (df["Tenure"] >= 5) & (df["IsActiveMember"] == 1)
    ).astype(int)

In [91]:
df["ProductDepth"] = pd.cut(
        df["NumOfProducts"],
        bins=[0, 1, 2, 4],
        labels=["Single", "Dual", "Multi"]
    )
 
    # --- 2.2 Is Multi-Product Customer ---
df["IsMultiProduct"] = (df["NumOfProducts"] >= 2).astype(int)
 
    # --- 2.3 Product-Activity Interaction ---
    # Captures customers who are active AND using multiple products
df["ProductActivityIndex"] = df["NumOfProducts"] * df["IsActiveMember"]
 
    # --- 2.4 Card + Active Combo ---
df["CardActiveCombo"] = (
        (df["HasCrCard"] == 1) & (df["IsActiveMember"] == 1)
    ).astype(int)
 
    # --- 2.5 Product Engagement Ratio ---
    # Normalizes product count by max possible (4 products)
df["ProductEngagementRatio"] = df["NumOfProducts"] / 4.0
 
    # --- 2.6 Credit Card Stickiness Score ---
    # Rewards card ownership combined with activity
df["CreditCardStickinessScore"] = (
        df["HasCrCard"] * 0.4 +
        df["IsActiveMember"] * 0.4 +
        (df["NumOfProducts"] / 4) * 0.2
    )
 

In [92]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler


balance_median = df["Balance"].median()

salary_median  = df["EstimatedSalary"].median()
 
    # --- 3.1 Balance Tiers ---
df["BalanceTier"] = pd.cut(
        df["Balance"],
        bins=[-1, 1, 50_000, 100_000, 150_000, df["Balance"].max() + 1],
        labels=["Zero", "Low", "Mid", "High", "Premium"]
    )
 
    # --- 3.2 Zero Balance Flag ---
df["IsZeroBalance"] = (df["Balance"] == 0).astype(int)
 
    # --- 3.3 Balance-to-Salary Ratio ---
df["BalanceToSalaryRatio"] = df["Balance"] / (df["EstimatedSalary"] + 1)
 
    # --- 3.4 Salary–Balance Mismatch ---
    # High salary but near-zero balance → possible disengagement signal
df["SalaryBalanceMismatch"] = (
        (df["EstimatedSalary"] > salary_median) &
        (df["Balance"] < balance_median * 0.25)
    ).astype(int)
 
    # --- 3.5 At-Risk Premium Customer ---
    # High balance + inactive = dangerous churn segment
df["AtRiskPremiumCustomer"] = (
        (df["Balance"] > balance_median) &
        (df["IsActiveMember"] == 0)
    ).astype(int)
 
    # --- 3.6 High Balance Active (Sticky Premium) ---
df["HighBalanceActive"] = (
        (df["Balance"] > balance_median) &
        (df["IsActiveMember"] == 1)
    ).astype(int)
 
    # --- 3.7 Salary Tier ---
df["SalaryTier"] = pd.qcut(
        df["EstimatedSalary"],
        q=4,
        labels=["Q1_Low", "Q2_Mid", "Q3_Upper", "Q4_High"]
    )
 
    # --- 3.8 Wealth Index ---
scaler = MinMaxScaler()
df["WealthIndex"] = scaler.fit_transform(
        df[["Balance", "EstimatedSalary"]].values
    ).mean(axis=1)

In [93]:
df["AgeBand"] = pd.cut(
        df["Age"],
        bins=[0, 25, 35, 45, 55, 65, 120],
        labels=["GenZ", "Millennial", "GenX_Early", "GenX_Late", "Boomer", "Senior"]
    )
 
    # --- 4.2 Senior Risk Flag (older customers churn differently) ---
df["IsSeniorRisk"] = (df["Age"] > 55).astype(int)
 
    # --- 4.3 Geography Encoding ---
geo_map = {"France": 0, "Spain": 1, "Germany": 2}
df["GeographyEncoded"] = df["Geography"].map(geo_map)
 
    # --- 4.4 Germany High-Balance Flag (Germany shows distinct churn patterns) ---
df["GermanyHighBalance"] = (
        (df["Geography"] == "Germany") &
        (df["Balance"] > df["Balance"].median())
    ).astype(int)
 
    # --- 4.5 Gender Encoded ---
df["GenderEncoded"] = LabelEncoder().fit_transform(df["Gender"])
 
    # --- 4.6 Credit Score Bands ---
df["CreditScoreBand"] = pd.cut(
        df["CreditScore"],
        bins=[0, 579, 669, 739, 799, 850],
        labels=["Poor", "Fair", "Good", "VeryGood", "Exceptional"]
    )
 
    # --- 4.7 Age × Tenure Interaction ---
df["AgeTenureProduct"] = df["Age"] * df["Tenure"]
 
    # --- 4.8 Young with Low Tenure (Early-Life Churn Risk) ---
df["YoungLowTenure"] = (
        (df["Age"] < 35) & (df["Tenure"] <= 2)
    ).astype(int)

In [94]:
 # --- KPI 1: Engagement Retention Ratio Score ---
    # Proxy per customer: active + tenure contribution
df["EngagementRetentionScore"] = (
        df["IsActiveMember"] * 0.5 +
        (df["Tenure"] / df["Tenure"].max()) * 0.3 +
        df["HasCrCard"] * 0.2
    )
 
    # --- KPI 2: Product Depth Index ---
df["ProductDepthIndex"] = (
        (df["NumOfProducts"] / 4) * 0.6 +
        df["IsActiveMember"] * 0.4
    )
 
    # --- KPI 3: High-Balance Disengagement Rate (per customer flag) ---
df["HighBalanceDisengaged"] = (
        (df["Balance"] > df["Balance"].quantile(0.75)) &
        (df["IsActiveMember"] == 0)
    ).astype(int)
 
    # --- KPI 4: Credit Card Stickiness Score (already computed in step 2) ---
    # df["CreditCardStickinessScore"] ← already exists
 
    # --- KPI 5: Relationship Strength Index (RSI) ---
    # Holistic score combining engagement, products, tenure, and balance
balance_norm  = MinMaxScaler().fit_transform(df[["Balance"]]).flatten()
salary_norm   = MinMaxScaler().fit_transform(df[["EstimatedSalary"]]).flatten()
tenure_norm   = df["Tenure"] / df["Tenure"].max()
credit_norm   = (df["CreditScore"] - df["CreditScore"].min()) / \
                    (df["CreditScore"].max() - df["CreditScore"].min())
 
df["RelationshipStrengthIndex"] = (
        df["IsActiveMember"]          * 0.25 +
        df["ProductDepthIndex"]       * 0.25 +
        tenure_norm                   * 0.20 +
        balance_norm                  * 0.15 +
        credit_norm                   * 0.10 +
        df["HasCrCard"]               * 0.05
    )
 
    # --- Sticky Customer Flag (RSI threshold) ---
rsi_threshold = df["RelationshipStrengthIndex"].quantile(0.70)
df["IsStickyCustomer"] = (
        df["RelationshipStrengthIndex"] >= rsi_threshold
    ).astype(int)

In [95]:
risk = pd.Series(np.zeros(len(df)), index=df.index)

risk += (df["IsActiveMember"] == 0).astype(int) * 2.0     # Inactive → +2
risk += (df["NumOfProducts"] == 1).astype(int) * 1.5      # Single product → +1.5
risk += (df["NumOfProducts"] > 2).astype(int) * 2.5       # 3–4 products → high churn (known pattern)
risk += df["IsZeroBalance"].astype(int) * 1.0             # Zero balance → +1
risk += df["AtRiskPremiumCustomer"].astype(int) * 2.0     # Premium + inactive → +2
risk += df["SalaryBalanceMismatch"].astype(int) * 1.0     # Mismatch → +1
risk += df["IsSeniorRisk"].astype(int) * 1.0              # Age > 55 → +1
risk += (df["Tenure"] <= 1).astype(int) * 1.0             # Very new → +1
risk += (df["CreditScore"] < 580).astype(int) * 0.5       # Poor credit → +0.5
risk += (df["GermanyHighBalance"]).astype(int) * 1.0      # Germany + high bal → +1
 
    # Normalize 0–10
df["RetentionRiskScore"] = (
        MinMaxScaler(feature_range=(0, 10))
        .fit_transform(risk.values.reshape(-1, 1))
        .flatten()
    )
 
df["RiskTier"] = pd.cut(
        df["RetentionRiskScore"],
        bins=[0, 3.33, 6.66, 10],
        labels=["Low Risk", "Medium Risk", "High Risk"],
        include_lowest=True
    )

In [96]:
categorical_cols = df.select_dtypes(include='object').columns
categorical_cols

Index(['Surname', 'Geography', 'Gender', 'EngagementSegment'], dtype='object')

In [97]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
numerical_cols

Index(['Year', 'CustomerId', 'CreditScore', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited', 'ActivityScore', 'IsLongTenureActive', 'IsMultiProduct',
       'ProductActivityIndex', 'CardActiveCombo', 'ProductEngagementRatio',
       'CreditCardStickinessScore', 'IsZeroBalance', 'BalanceToSalaryRatio',
       'SalaryBalanceMismatch', 'AtRiskPremiumCustomer', 'HighBalanceActive',
       'WealthIndex', 'IsSeniorRisk', 'GeographyEncoded', 'GermanyHighBalance',
       'GenderEncoded', 'AgeTenureProduct', 'YoungLowTenure',
       'EngagementRetentionScore', 'ProductDepthIndex',
       'HighBalanceDisengaged', 'RelationshipStrengthIndex',
       'IsStickyCustomer', 'RetentionRiskScore'],
      dtype='object')

In [98]:
pd.options.display.max_columns = 500

df

,Year,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,EngagementSegment,ActivityScore,TenureBand,IsLongTenureActive,ProductDepth,IsMultiProduct,ProductActivityIndex,CardActiveCombo,ProductEngagementRatio,CreditCardStickinessScore,BalanceTier,IsZeroBalance,BalanceToSalaryRatio,SalaryBalanceMismatch,AtRiskPremiumCustomer,HighBalanceActive,SalaryTier,WealthIndex,AgeBand,IsSeniorRisk,GeographyEncoded,GermanyHighBalance,GenderEncoded,CreditScoreBand,AgeTenureProduct,YoungLowTenure,EngagementRetentionScore,ProductDepthIndex,HighBalanceDisengaged,RelationshipStrengthIndex,IsStickyCustomer,RetentionRiskScore,RiskTier
0,2025,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1,Active_LowProduct,4,Early (2-3y),0,Single,0,1,1,0.25,0.85,Zero,1,0.000000,1,0,0,Q3_Upper,0.253367,GenX_Early,0,0,0,0,Fair,84,0,0.76,0.55,0,0.531300,0,3.684211,Medium Risk
1,2025,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,Active_LowProduct,3,New (0-1y),0,Single,0,1,0,0.25,0.45,Mid,0,0.744670,0,0,0,Q3_Upper,0.448370,GenX_Early,0,1,0,0,Fair,41,0,0.53,0.55,0,0.509205,0,2.631579,Low Risk
2,2025,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,Inactive_HighBalance,3,Loyal (7y+),0,Multi,1,0,0,0.75,0.55,Premium,0,1.401362,0,1,0,Q3_Upper,0.603006,GenX_Early,0,0,0,0,Poor,336,0,0.44,0.45,1,0.448354,0,7.368421,High Risk
3,2025,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0,Other,2,New (0-1y),0,Dual,1,0,0,0.50,0.10,Zero,1,0.000000,0,0,0,Q2_Mid,0.234560,GenX_Early,0,0,0,0,Good,39,0,0.03,0.30,0,0.164800,0,4.210526,Medium Risk
4,2025,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,Active_LowProduct,4,Early (2-3y),0,Single,0,1,1,0.25,0.85,High,0,1.587035,0,0,1,Q2_Mid,0.447823,GenX_Early,0,1,0,0,Exceptional,86,0,0.76,0.55,0,0.652537,1,1.578947,Low Risk
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2025,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0,Other,3,Mid (4-6y),0,Dual,1,0,0,0.50,0.50,Zero,1,0.000000,0,0,0,Q2_Mid,0.240671,GenX_Early,0,0,0,1,VeryGood,195,0,0.35,0.30,0,0.309200,0,3.157895,Low Risk
9996,2025,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0,Active_LowProduct,4,Loyal (7y+),1,Single,0,1,1,0.25,0.85,Mid,0,0.564102,0,0,0,Q3_Upper,0.368573,Millennial,0,0,0,1,Poor,350,0,1.00,0.55,0,0.704999,1,2.105263,Low Risk
9997,2025,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1,Active_LowProduct,3,Loyal (7y+),1,Single,0,1,0,0.25,0.45,Zero,1,0.000000,0,0,0,Q1_Low,0.105195,GenX_Early,0,0,0,0,Good,252,0,0.71,0.55,0,0.599300,0,2.631579,Low Risk
9998,2025,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1,Other,3,Early (2-3y),0,Dual,1,0,0,0.50,0.50,Mid,0,0.808222,0,0,0,Q2_Mid,0.381828,GenX_Early,0,2,0,1,VeryGood,126,0,0.29,0.30,0,0.314284,0,2.105263,Low Risk


In [99]:
df.drop('Surname', axis=1, inplace=True)

In [100]:
# Binary : Gender 

df['Gender']=df['Gender'].apply(lambda x: 1 if x == 'Male' else 0)

In [101]:
# ── Ordinal Encoding for naturally-ordered categorical features ──
# These features have a clear ranking, so we map them to integers (0, 1, 2 …)

# TenureBand  →  0=New, 1=Early, 2=Mid, 3=Loyal
tenure_order = {'New (0-1y)': 0, 'Early (2-3y)': 1, 'Mid (4-6y)': 2, 'Loyal (7y+)': 3}
df['TenureBand'] = df['TenureBand'].astype(str).map(tenure_order)

Salary_order = {'Q1_Low': 0, 'Q2_Mid': 1, 'Q3_Upper': 2, 'Q4_High': 3}
df['SalaryTier'] = df['SalaryTier'].astype(str).map(Salary_order)


Geograpy_order = {'France': 0, 'Spain': 1, 'Germany': 2}
df['Geography'] = df['Geography'].astype(str).map(Geograpy_order)

# ProductDepth  →  0=Single, 1=Dual, 2=Multi
product_order = {'Single': 0, 'Dual': 1, 'Multi': 2}
df['ProductDepth'] = df['ProductDepth'].astype(str).map(product_order)

# BalanceTier  →  0=Zero, 1=Low, 2=Mid, 3=High, 4=Premium
balance_order = {'Zero': 0, 'Low': 1, 'Mid': 2, 'High': 3, 'Premium': 4}
df['BalanceTier'] = df['BalanceTier'].astype(str).map(balance_order)

# CreditScoreBand  →  0=Poor, 1=Fair, 2=Good, 3=VeryGood, 4=Exceptional
credit_order = {'Poor': 0, 'Fair': 1, 'Good': 2, 'VeryGood': 3, 'Exceptional': 4}
df['CreditScoreBand'] = df['CreditScoreBand'].astype(str).map(credit_order)

# RiskTier  →  0=Low Risk, 1=Medium Risk, 2=High Risk
risk_order = {'Low Risk': 0, 'Medium Risk': 1, 'High Risk': 2}
df['RiskTier'] = df['RiskTier'].astype(str).map(risk_order)

print('Ordinal encoding done.')
df[['TenureBand','ProductDepth','BalanceTier','CreditScoreBand','RiskTier']].head()

Ordinal encoding done.


,TenureBand,ProductDepth,BalanceTier,CreditScoreBand,RiskTier
0,1,0,0,1,1
1,0,0,2,1,0
2,3,2,4,0,2
3,0,1,0,2,1
4,1,0,3,4,0


In [102]:
# ── One-Hot Encoding for nominal categorical features ──
# AgeBand has no numeric order → explode into binary (0/1) dummy columns
# drop_first=True drops one dummy per group to avoid multicollinearity

df = pd.get_dummies(df, columns=['AgeBand'], prefix='AgeBand', drop_first=False, dtype=int)

# Also one-hot encode EngagementSegment and SalaryTier if still present
for col in ['EngagementSegment', 'SalaryTier']:
    if col in df.columns and df[col].dtype == object:
        df = pd.get_dummies(df, columns=[col], prefix=col, drop_first=False, dtype=int)

print('One-hot encoding done.')
# Show only newly created dummy columns
dummy_cols = [c for c in df.columns if c.startswith(('AgeBand_','EngagementSegment_','SalaryTier_'))]
df[dummy_cols].head()

One-hot encoding done.


,AgeBand_GenZ,AgeBand_Millennial,AgeBand_GenX_Early,AgeBand_GenX_Late,AgeBand_Boomer,AgeBand_Senior,EngagementSegment_Active_Engaged,EngagementSegment_Active_LowProduct,EngagementSegment_Inactive_Disengaged,EngagementSegment_Inactive_HighBalance,EngagementSegment_Other
0,0,0,1,0,0,0,0,1,0,0,0
1,0,0,1,0,0,0,0,1,0,0,0
2,0,0,1,0,0,0,0,0,0,1,0
3,0,0,1,0,0,0,0,0,0,0,1
4,0,0,1,0,0,0,0,1,0,0,0


In [103]:
df

,Year,CustomerId,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,ActivityScore,TenureBand,IsLongTenureActive,ProductDepth,IsMultiProduct,ProductActivityIndex,CardActiveCombo,ProductEngagementRatio,CreditCardStickinessScore,BalanceTier,IsZeroBalance,BalanceToSalaryRatio,SalaryBalanceMismatch,AtRiskPremiumCustomer,HighBalanceActive,SalaryTier,WealthIndex,IsSeniorRisk,GeographyEncoded,GermanyHighBalance,GenderEncoded,CreditScoreBand,AgeTenureProduct,YoungLowTenure,EngagementRetentionScore,ProductDepthIndex,HighBalanceDisengaged,RelationshipStrengthIndex,IsStickyCustomer,RetentionRiskScore,RiskTier,AgeBand_GenZ,AgeBand_Millennial,AgeBand_GenX_Early,AgeBand_GenX_Late,AgeBand_Boomer,AgeBand_Senior,EngagementSegment_Active_Engaged,EngagementSegment_Active_LowProduct,EngagementSegment_Inactive_Disengaged,EngagementSegment_Inactive_HighBalance,EngagementSegment_Other
0,2025,15634602,619,0,0,42,2,0.00,1,1,1,101348.88,1,4,1,0,0,0,1,1,0.25,0.85,0,1,0.000000,1,0,0,2,0.253367,0,0,0,0,1,84,0,0.76,0.55,0,0.531300,0,3.684211,1,0,0,1,0,0,0,0,1,0,0,0
1,2025,15647311,608,1,0,41,1,83807.86,1,0,1,112542.58,0,3,0,0,0,0,1,0,0.25,0.45,2,0,0.744670,0,0,0,2,0.448370,0,1,0,0,1,41,0,0.53,0.55,0,0.509205,0,2.631579,0,0,0,1,0,0,0,0,1,0,0,0
2,2025,15619304,502,0,0,42,8,159660.80,3,1,0,113931.57,1,3,3,0,2,1,0,0,0.75,0.55,4,0,1.401362,0,1,0,2,0.603006,0,0,0,0,0,336,0,0.44,0.45,1,0.448354,0,7.368421,2,0,0,1,0,0,0,0,0,0,1,0
3,2025,15701354,699,0,0,39,1,0.00,2,0,0,93826.63,0,2,0,0,1,1,0,0,0.50,0.10,0,1,0.000000,0,0,0,1,0.234560,0,0,0,0,2,39,0,0.03,0.30,0,0.164800,0,4.210526,1,0,0,1,0,0,0,0,0,0,0,1
4,2025,15737888,850,1,0,43,2,125510.82,1,1,1,79084.10,0,4,1,0,0,0,1,1,0.25,0.85,3,0,1.587035,0,0,1,1,0.447823,0,1,0,0,4,86,0,0.76,0.55,0,0.652537,1,1.578947,0,0,0,1,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,2025,15606229,771,0,1,39,5,0.00,2,1,0,96270.64,0,3,2,0,1,1,0,0,0.50,0.50,0,1,0.000000,0,0,0,1,0.240671,0,0,0,1,3,195,0,0.35,0.30,0,0.309200,0,3.157895,0,0,0,1,0,0,0,0,0,0,0,1
9996,2025,15569892,516,0,1,35,10,57369.61,1,1,1,101699.77,0,4,3,1,0,0,1,1,0.25,0.85,2,0,0.564102,0,0,0,2,0.368573,0,0,0,1,0,350,0,1.00,0.55,0,0.704999,1,2.105263,0,0,1,0,0,0,0,0,1,0,0,0
9997,2025,15584532,709,0,0,36,7,0.00,1,0,1,42085.58,1,3,3,1,0,0,1,0,0.25,0.45,0,1,0.000000,0,0,0,0,0.105195,0,0,0,0,2,252,0,0.71,0.55,0,0.599300,0,2.631579,0,0,0,1,0,0,0,0,1,0,0,0
9998,2025,15682355,772,2,1,42,3,75075.31,2,1,0,92888.52,1,3,1,0,1,1,0,0,0.50,0.50,2,0,0.808222,0,0,0,1,0.381828,0,2,0,1,3,126,0,0.29,0.30,0,0.314284,0,2.105263,0,0,0,1,0,0,0,0,0,0,0,1


In [104]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

def handle_imbalance(X, y):

    # Step 1: Train-Test Split (with stratification)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    print("Before SMOTE:")
    print(y_train.value_counts())

    # Step 2: Apply SMOTE ONLY on training data
    smote = SMOTE(random_state=42)

    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    print("\nAfter SMOTE:")
    print(y_train_resampled.value_counts())

    return X_train_resampled, X_test, y_train_resampled, y_test

In [85]:
df['SalaryTier'].unique()

['Q3_Upper', 'Q2_Mid', 'Q4_High', 'Q1_Low']
Categories (4, object): ['Q1_Low' < 'Q2_Mid' < 'Q3_Upper' < 'Q4_High']

In [105]:
X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = handle_imbalance(X, y)

Before SMOTE:
Exited
0    6370
1    1630
Name: count, dtype: int64

After SMOTE:
Exited
1    6370
0    6370
Name: count, dtype: int64


In [107]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

def handle_imbalance(X, y):

    # 🔹 Full dataset distribution
    print("📊 Full Dataset Distribution:")
    print(y.value_counts())

    # 🔹 Train-Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        stratify=y,
        random_state=42
    )

    print("\n📊 Train Set Before SMOTE:")
    print(y_train.value_counts())

    print("\n📊 Test Set (Unchanged):")
    print(y_test.value_counts())

    # 🔹 Apply SMOTE
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

    print("\n📊 Train Set After SMOTE:")
    print(y_train_resampled.value_counts())

    return X_train_resampled, X_test, y_train_resampled, y_test


# 🔥 Main Execution
X = df.drop('Exited', axis=1)
y = df['Exited']

X_train, X_test, y_train, y_test = handle_imbalance(X, y)

📊 Full Dataset Distribution:
Exited
0    7963
1    2037
Name: count, dtype: int64

📊 Train Set Before SMOTE:
Exited
0    6370
1    1630
Name: count, dtype: int64

📊 Test Set (Unchanged):
Exited
0    1593
1     407
Name: count, dtype: int64

📊 Train Set After SMOTE:
Exited
1    6370
0    6370
Name: count, dtype: int64


In [108]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
    )

In [110]:
X = df.drop(columns=['Exited'],axis=1)

In [111]:
y = df['Exited']

In [112]:
y

0       1
1       0
2       1
3       0
4       0
       ..
9995    0
9996    0
9997    1
9998    1
9999    0
Name: Exited, Length: 10000, dtype: int64

In [113]:
# Create Column Transformer with 3 types of transformers
num_features = X.select_dtypes(exclude="object").columns
cat_features = X.select_dtypes(include="object").columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
         ("StandardScaler", numeric_transformer, num_features),        
    ]
)

In [114]:
X = preprocessor.fit_transform(X)

In [115]:
# separate dataset into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape, X_test.shape

((8000, 54), (2000, 54))

In [116]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

In [117]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(probability=True),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

In [118]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

results = []

for name, model in models.items():
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    
    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)
    
    results.append([name, acc, prec, rec, f1, roc])

    print(f"\n{name}")
    print(f"Accuracy: {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall: {rec:.3f}")
    print(f"F1 Score: {f1:.3f}")
    print(f"ROC-AUC: {roc:.3f}")


Logistic Regression
Accuracy: 0.855
Precision: 0.691
Recall: 0.473
F1 Score: 0.562
ROC-AUC: 0.854

Decision Tree
Accuracy: 0.778
Precision: 0.443
Recall: 0.501
F1 Score: 0.470
ROC-AUC: 0.673

Random Forest
Accuracy: 0.867
Precision: 0.750
Recall: 0.481
F1 Score: 0.586
ROC-AUC: 0.855

Gradient Boosting
Accuracy: 0.860
Precision: 0.722
Recall: 0.468
F1 Score: 0.568
ROC-AUC: 0.869

KNN
Accuracy: 0.835
Precision: 0.608
Recall: 0.443
F1 Score: 0.513
ROC-AUC: 0.785

SVM
Accuracy: 0.855
Precision: 0.745
Recall: 0.394
F1 Score: 0.516
ROC-AUC: 0.826


c:\Users\nishi\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [02:08:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



XGBoost
Accuracy: 0.855
Precision: 0.681
Recall: 0.489
F1 Score: 0.569
ROC-AUC: 0.842


In [119]:
import pandas as pd

results_df = pd.DataFrame(results, columns=[
    "Model", "Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"
])

results_df.sort_values(by="ROC-AUC", ascending=False)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
3,Gradient Boosting,0.8600,0.721569,0.468193,0.567901,0.868809
2,Random Forest,0.8665,0.750000,0.480916,0.586047,0.854577
0,Logistic Regression,0.8550,0.691450,0.473282,0.561934,0.854428
6,XGBoost,0.8545,0.680851,0.488550,0.568889,0.841664
5,SVM,0.8545,0.745192,0.394402,0.515807,0.825811
4,KNN,0.8345,0.608392,0.442748,0.512518,0.784524
1,Decision Tree,0.7780,0.442697,0.501272,0.470167,0.673474


In [120]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, ConfusionMatrixDisplay
)


def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """Train, predict, and return a metrics dict."""
    model.fit(X_tr, y_tr)
    y_pred      = model.predict(X_te)
    y_proba     = model.predict_proba(X_te)[:, 1]

    metrics = {
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_te, y_pred),  4),
        'Precision': round(precision_score(y_te, y_pred), 4),
        'Recall'   : round(recall_score(y_te, y_pred),    4),
        'F1-Score' : round(f1_score(y_te, y_pred),        4),
        'ROC-AUC'  : round(roc_auc_score(y_te, y_proba),  4),
    }

    print(f'\n{"─"*50}')
    print(f'  {name}')
    print(f'{"─"*50}')
    for k, v in metrics.items():
        if k != 'Model':
            print(f'  {k:<12}: {v}')
    print(classification_report(y_te, y_pred, target_names=['No Attrition', 'Attrition']))

    return model, metrics, y_pred, y_proba

print('Evaluation helper defined.')

Evaluation helper defined.


In [121]:
lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

lr_model, lr_metrics, lr_pred, lr_proba = evaluate_model(
    'Logistic Regression', lr, X_train, y_train, X_test, y_test
)


──────────────────────────────────────────────────
  Logistic Regression
──────────────────────────────────────────────────
  Accuracy    : 0.7875
  Precision   : 0.4752
  Recall      : 0.7812
  F1-Score    : 0.591
  ROC-AUC     : 0.8546
              precision    recall  f1-score   support

No Attrition       0.94      0.79      0.86      1607
   Attrition       0.48      0.78      0.59       393

    accuracy                           0.79      2000
   macro avg       0.71      0.79      0.72      2000
weighted avg       0.85      0.79      0.80      2000



In [122]:
rf_param_grid = {
    'n_estimators'     : [100, 200, 300],
    'max_depth'        : [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'max_features'     : ['sqrt', 'log2'],
    'class_weight'     : ['balanced'],
}

rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_search = RandomizedSearchCV(
    rf_base,
    rf_param_grid,
    n_iter=30,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rf_search.fit(X_train, y_train)
print(f'\nBest RF params: {rf_search.best_params_}')
print(f'Best CV ROC-AUC: {rf_search.best_score_:.4f}')

rf_model, rf_metrics, rf_pred, rf_proba = evaluate_model(
    'Random Forest', rf_search.best_estimator_, X_train, y_train, X_test, y_test
)

Fitting 5 folds for each of 30 candidates, totalling 150 fits

Best RF params: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 10, 'class_weight': 'balanced'}
Best CV ROC-AUC: 0.8634

──────────────────────────────────────────────────
  Random Forest
──────────────────────────────────────────────────
  Accuracy    : 0.827
  Precision   : 0.5451
  Recall      : 0.7226
  F1-Score    : 0.6214
  ROC-AUC     : 0.8625
              precision    recall  f1-score   support

No Attrition       0.93      0.85      0.89      1607
   Attrition       0.55      0.72      0.62       393

    accuracy                           0.83      2000
   macro avg       0.74      0.79      0.75      2000
weighted avg       0.85      0.83      0.84      2000



In [123]:
# Compute scale_pos_weight for XGBoost's internal imbalance handling
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / pos
print(f'scale_pos_weight: {scale_pos_weight:.2f}')

xgb_param_grid = {
    'n_estimators'      : [100, 200, 300],
    'max_depth'         : [3, 5, 7],
    'learning_rate'     : [0.01, 0.05, 0.1, 0.2],
    'subsample'         : [0.7, 0.8, 1.0],
    'colsample_bytree'  : [0.7, 0.8, 1.0],
    'reg_alpha'         : [0, 0.1, 0.5],
    'reg_lambda'        : [1, 1.5, 2],
}

xgb_base = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

xgb_search = RandomizedSearchCV(
    xgb_base,
    xgb_param_grid,
    n_iter=30,
    scoring='roc_auc',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

xgb_search.fit(X_train, y_train)
print(f'\nBest XGB params: {xgb_search.best_params_}')
print(f'Best CV ROC-AUC: {xgb_search.best_score_:.4f}')

xgb_model, xgb_metrics, xgb_pred, xgb_proba = evaluate_model(
    'XGBoost', xgb_search.best_estimator_, X_train, y_train, X_test, y_test
)

scale_pos_weight: 3.87
Fitting 5 folds for each of 30 candidates, totalling 150 fits


c:\Users\nishi\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [02:14:55] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



Best XGB params: {'subsample': 0.7, 'reg_lambda': 1, 'reg_alpha': 0.5, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 1.0}
Best CV ROC-AUC: 0.8636


c:\Users\nishi\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [02:14:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



──────────────────────────────────────────────────
  XGBoost
──────────────────────────────────────────────────
  Accuracy    : 0.81
  Precision   : 0.5111
  Recall      : 0.7634
  F1-Score    : 0.6122
  ROC-AUC     : 0.8681
              precision    recall  f1-score   support

No Attrition       0.93      0.82      0.87      1607
   Attrition       0.51      0.76      0.61       393

    accuracy                           0.81      2000
   macro avg       0.72      0.79      0.74      2000
weighted avg       0.85      0.81      0.82      2000



In [129]:
results_df = pd.DataFrame([lr_metrics, rf_metrics, xgb_metrics])
results_df = results_df.set_index('Model')

print('\n===== MODEL COMPARISON =====')
print(results_df.to_string())

# Highlight best value per metric
results_df.style.highlight_max(axis=0, color='#d4edda')


===== MODEL COMPARISON =====
                     Accuracy  Precision  Recall  F1-Score  ROC-AUC
Model                                                              
Logistic Regression    0.7875     0.4752  0.7812    0.5910   0.8546
Random Forest          0.8270     0.5451  0.7226    0.6214   0.8625
XGBoost                0.8100     0.5111  0.7634    0.6122   0.8681


,Accuracy,Precision,Recall,F1-Score,ROC-AUC
Model,,,,,
Logistic Regression,0.787500,0.475200,0.781200,0.591000,0.854600
Random Forest,0.827000,0.545100,0.722600,0.621400,0.862500
XGBoost,0.810000,0.511100,0.763400,0.612200,0.868100


In [130]:
# Pick best model by ROC-AUC
best_row = results_df['Recall'].idxmax()
print(f'Best model by Recall: {best_row}')

model_map = {
    'Logistic Regression': lr_model,
    'Random Forest'      : rf_model,
    'XGBoost'            : xgb_model,
}

best_model= model_map[best_row]
best_model_proba = {
    'Logistic Regression': lr_proba,
    'Random Forest'      : rf_proba,
    'XGBoost'            : xgb_proba,
}[best_row]

joblib.dump(best_model, 'artifacts/best_model.pkl')
joblib.dump(results_df, 'artifacts/results_df.pkl')
np.save('artifacts/best_proba.npy', best_model_proba)

print('Saved: artifacts/best_model.pkl')
print('Saved: artifacts/results_df.pkl')
print('Saved: artifacts/best_proba.npy')
print(f'\nFinal metrics of best model ({best_row}):')
print(results_df.loc[best_row])

Best model by Recall: Logistic Regression


FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/best_model.pkl'